# Clase 04 – Data Science I

| # | Tema |
|---|------|
| 1 | Pipelines reproducibles |
| 2 | Valores ausentes – teoría (MCAR/MAR/MNAR) |
| 3 | Imputación básica (dropna, fillna, group-based) |
| 4 | Imputación avanzada (SimpleImputer, IterativeImputer, ColumnTransformer) |
| 5 | Series de tiempo – parsing y slicing de fechas |
| 6 | Zonas horarias y shift/lag |
| 7 | Resampling, rolling, EWMA y expanding |
| 8 | GroupBy y agregaciones |
| 9 | Operaciones sobre strings |
| 10 | Tipos de datos, category y downcast |
| 11 | loc / iloc y máscaras booleanas |
| 12 | Transformaciones vectorizadas (np.where, assign, np.select) |
| 13 | Matplotlib – principios de visualización |
| 14 | Matplotlib – tipos de gráficos |
| 15 | Plotly Express – gráficos interactivos |
| 16 | Storytelling con datos |

## Setup – Importaciones

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
try:
    from sklearn.impute import IterativeImputer
except ImportError:
    from sklearn.experimental import enable_iterative_imputer  # noqa
    from sklearn.impute import IterativeImputer
try:
    import plotly.express as px
    PLOTLY = True
except ImportError:
    PLOTLY = False
    print('plotly no instalado – pip install plotly')

pd.set_option('display.max_columns', 20)
pd.set_option('display.float_format', '{:.3f}'.format)
np.random.seed(42)
print('Librerías cargadas correctamente')

---
## Unidad 1 – Pipelines Reproducibles

Un **pipeline** encadena pasos de preprocesamiento y modelado en un único
objeto que puede ajustarse, evaluarse y serializarse.

**Ventajas**
- Evita *data leakage*: la transformación se ajusta solo sobre el train set.
- Facilita la búsqueda de hiperparámetros con `GridSearchCV`.
- Serializable con `joblib.dump` → despliegue reproducible.

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split

# Dataset sintético
rng = np.random.default_rng(0)
n = 300
df_pipe = pd.DataFrame({
    'x1': rng.normal(10, 3, n),
    'x2': rng.normal(5,  2, n),
    'x3': rng.normal(0,  1, n),
})
df_pipe['y'] = 2*df_pipe['x1'] - df_pipe['x2'] + rng.normal(0, 1, n)

# Introducir NaN artificiales
for col in ['x1','x2','x3']:
    idx = rng.choice(n, size=15, replace=False)
    df_pipe.loc[idx, col] = np.nan

X = df_pipe[['x1','x2','x3']]
y = df_pipe['y']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='mean')),
    ('scaler',  StandardScaler()),
    ('model',   LinearRegression()),
])

pipeline.fit(X_train, y_train)
score = pipeline.score(X_test, y_test)
print(f'R² en test: {score:.4f}')
print('Pasos del pipeline:', [s for s, _ in pipeline.steps])

In [ ]:
# Inspeccionar coeficientes
modelo = pipeline.named_steps['model']
features = ['x1','x2','x3']
for feat, coef in zip(features, modelo.coef_):
    print(f'  {feat}: {coef:.4f}')

---
## Unidad 2 – Valores Ausentes: Teoría

| Tipo | Definición | Ejemplo |
|------|-----------|---------|
| **MCAR** | Missing Completely At Random | Sensor defectuoso aleatorio |
| **MAR** | Missing At Random | Mayores no reportan edad |
| **MNAR** | Missing Not At Random | Pacientes graves no responden |

### ¿Por qué importa?
- MCAR → imputar o eliminar sin sesgo.
- MAR → imputar condicionado a otras variables.
- MNAR → requiere modelos específicos o recolección adicional.

In [ ]:
# Diagnóstico de valores ausentes
rng = np.random.default_rng(1)
df_nan = pd.DataFrame({
    'edad':    rng.choice([np.nan, 25, 30, 45, 50, 60], size=100, p=[0.1,0.18,0.18,0.18,0.18,0.18]),
    'ingreso': rng.choice([np.nan, 20000, 35000, 50000, 80000], size=100, p=[0.15,0.21,0.21,0.21,0.22]),
    'score':   rng.choice([np.nan, 1, 2, 3, 4, 5], size=100, p=[0.05,0.19,0.19,0.19,0.19,0.19]),
    'ciudad':  rng.choice([np.nan,'BA','Córdoba','Rosario'], size=100, p=[0.08,0.31,0.30,0.31]),
})

print('--- % de nulos por columna ---')
print((df_nan.isnull().mean()*100).round(1).to_string())
print()
print('--- Resumen isnull().sum() ---')
print(df_nan.isnull().sum())

In [ ]:
# Heatmap de nulos (seaborn)
fig, ax = plt.subplots(figsize=(7, 3))
sns.heatmap(df_nan.isnull().T, cbar=False, cmap='viridis', ax=ax)
ax.set_title('Mapa de valores ausentes')
ax.set_xlabel('Observaciones')
plt.tight_layout()
plt.show()

In [ ]:
# Columna indicadora de ausencia (missing indicator)
df_nan['ingreso_missing'] = df_nan['ingreso'].isnull().astype(int)
print(df_nan[['ingreso','ingreso_missing']].head(10))

In [ ]:
# Distribución de edad según si ingreso es nulo
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
df_nan[df_nan['ingreso_missing']==0]['edad'].dropna().hist(ax=axes[0], bins=10, color='steelblue')
axes[0].set_title('Edad | ingreso presente')
df_nan[df_nan['ingreso_missing']==1]['edad'].dropna().hist(ax=axes[1], bins=10, color='salmon')
axes[1].set_title('Edad | ingreso ausente')
plt.tight_layout()
plt.show()

---
## Unidad 3 – Imputación Básica

| Método | Cuándo usarlo |
|--------|--------------|
| `dropna()` | Pocos nulos, sin sesgo si son MCAR |
| `fillna(valor)` | Rellenar con constante conocida |
| `fillna(media/mediana/moda)` | Distribución aproximadamente normal |
| `fillna(ffill/bfill)` | Series temporales |
| **Group-based** | Distribución varía por grupo |

In [ ]:
# dropna básico
df_clean = df_nan.dropna(subset=['edad','score'])
print(f'Filas originales: {len(df_nan)} | Tras dropna: {len(df_clean)}')

In [ ]:
# fillna con estadísticos
df_fill = df_nan.copy()
df_fill['edad']    = df_fill['edad'].fillna(df_fill['edad'].median())
df_fill['ingreso'] = df_fill['ingreso'].fillna(df_fill['ingreso'].mean())
df_fill['score']   = df_fill['score'].fillna(df_fill['score'].mode()[0])
df_fill['ciudad']  = df_fill['ciudad'].fillna(df_fill['ciudad'].mode()[0])
print('Nulos restantes:', df_fill.isnull().sum().to_dict())

In [ ]:
# Imputación basada en grupo (group-based)
rng = np.random.default_rng(2)
df_prod = pd.DataFrame({
    'categoria': rng.choice(['A','B','C'], size=120),
    'precio':    rng.uniform(10, 500, 120),
})
idx_nan = rng.choice(120, size=30, replace=False)
df_prod.loc[idx_nan, 'precio'] = np.nan

# Imputa con la mediana de cada categoría
df_prod['precio'] = df_prod['precio'].fillna(
    df_prod.groupby('categoria')['precio'].transform('median')
)
print('Nulos tras imputación group-based:', df_prod['precio'].isnull().sum())
print(df_prod.groupby('categoria')['precio'].describe().round(2))

---
## Unidad 4 – Imputación Avanzada (sklearn)

### SimpleImputer – 4 estrategias
| Estrategia | Tipo de columna |
|------------|----------------|
| `mean` | Numérica, distribución simétrica |
| `median` | Numérica, con outliers |
| `most_frequent` | Numérica o categórica |
| `constant` | Cualquiera |

### IterativeImputer (MICE)
Modela cada variable con NaN como función de las demás → imputación multivariante.

In [ ]:
# SimpleImputer aplicado a múltiples columnas
from sklearn.impute import SimpleImputer

X_num = df_nan[['edad','ingreso','score']].values

imp_median = SimpleImputer(strategy='median')
X_imputed  = imp_median.fit_transform(X_num)

df_imputed = pd.DataFrame(X_imputed, columns=['edad','ingreso','score'])
print('SimpleImputer(median) – primeras filas:')
print(df_imputed.head())

In [ ]:
# IterativeImputer (MICE multivariante)
imp_iter = IterativeImputer(max_iter=10, random_state=0)
X_mice   = imp_iter.fit_transform(X_num)

df_mice = pd.DataFrame(X_mice, columns=['edad','ingreso','score'])
print('IterativeImputer – primeras filas:')
print(df_mice.head())

In [ ]:
# ColumnTransformer: estrategia diferenciada por tipo
rng = np.random.default_rng(3)
n = 200
df_ct = pd.DataFrame({
    'precio':   rng.uniform(100, 5000, n),
    'cantidad': rng.integers(1, 100, n).astype(float),
    'categoria': rng.choice(['X','Y','Z','W'], n),
})
for col, pct in [('precio',0.10),('cantidad',0.08),('categoria',0.12)]:
    idx = rng.choice(n, int(n*pct), replace=False)
    df_ct.loc[idx, col] = np.nan

num_cols = ['precio','cantidad']
cat_cols = ['categoria']

preprocessor = ColumnTransformer(transformers=[
    ('num', SimpleImputer(strategy='median'),        num_cols),
    ('cat', SimpleImputer(strategy='most_frequent'), cat_cols),
])

arr = preprocessor.fit_transform(df_ct)
print('Shape tras ColumnTransformer:', arr.shape)
print('Nulos en resultado:', np.isnan(arr).sum())

In [ ]:
# Pipeline completo: imputación + escalado + modelo
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.model_selection import cross_val_score

X_ct = df_ct[num_cols + cat_cols]
y_ct = rng.normal(500, 100, n)  # target sintético

# Solo columnas numéricas para el ejemplo
full_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler',  StandardScaler()),
    ('ridge',   Ridge(alpha=1.0)),
])

scores = cross_val_score(full_pipe, df_ct[num_cols], y_ct, cv=5, scoring='r2')
print(f'R² cross-val: {scores.mean():.4f} ± {scores.std():.4f}')

### missingno – Visualización de patrones de ausencia

```python
import missingno as msno
msno.matrix(df)     # patrón de nulos
msno.bar(df)        # % de completitud por columna
msno.heatmap(df)    # correlación de nulidad
```
*(Requiere: `pip install missingno`)*

In [ ]:
try:
    import missingno as msno
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    msno.matrix(df_nan, ax=axes[0], sparkline=False)
    msno.bar(df_nan, ax=axes[1])
    plt.tight_layout()
    plt.show()
except ImportError:
    print('missingno no instalado. Ejecutar: pip install missingno')
    # Alternativa con seaborn
    fig, ax = plt.subplots(figsize=(6, 3))
    completitud = (1 - df_nan.isnull().mean()) * 100
    completitud.plot(kind='bar', ax=ax, color='steelblue', edgecolor='white')
    ax.set_title('% Completitud por columna')
    ax.set_ylabel('%')
    ax.set_ylim(0, 110)
    plt.tight_layout()
    plt.show()

---
## Unidad 5 – Series de Tiempo: Parsing y Slicing de Fechas

| Función | Propósito |
|---------|-----------|
| `pd.to_datetime(errors='coerce')` | Parsea, convierte inválidos a NaT |
| `pd.date_range(start, periods, freq)` | Genera rango de fechas |
| `pd.period_range(start, periods, freq)` | Genera rango de períodos |
| `set_index(col_fecha)` | Usa fecha como índice para slicing |
| `.loc['2023-01':'2023-06']` | Slice por rango de fechas (inclusive) |

In [ ]:
# Parsing con errors='coerce'
fechas_raw = ['2023-01-15', '2023-02-30', 'no-es-fecha', '2023-03-10', None]
serie_fechas = pd.to_datetime(fechas_raw, errors='coerce')
print(serie_fechas)
print(f'NaT generados: {serie_fechas.isna().sum()}')

In [ ]:
# date_range y period_range
rango_diario = pd.date_range(start='2023-01-01', periods=10, freq='D')
print('date_range diario:')
print(rango_diario)

rango_mensual = pd.period_range(start='2023-01', periods=6, freq='M')
print('\nperiod_range mensual:')
print(rango_mensual)

In [ ]:
# Serie con DatetimeIndex – slicing con .loc
rng = np.random.default_rng(4)
idx = pd.date_range('2022-01-01', periods=365, freq='D')
serie = pd.Series(rng.normal(100, 15, 365), index=idx, name='ventas')

# Slice por mes
enero = serie.loc['2022-01']
print(f'Enero 2022: {len(enero)} días, media={enero.mean():.2f}')

# Slice por rango
q1 = serie.loc['2022-01-01':'2022-03-31']
print(f'Q1 2022: {len(q1)} días, media={q1.mean():.2f}')

In [ ]:
# Extraer componentes temporales
df_ts = pd.DataFrame({'fecha': idx, 'ventas': serie.values})
df_ts['año']        = df_ts['fecha'].dt.year
df_ts['mes']        = df_ts['fecha'].dt.month
df_ts['dia_semana'] = df_ts['fecha'].dt.dayofweek   # 0=Lunes
df_ts['trimestre']  = df_ts['fecha'].dt.quarter
print(df_ts.head())

---
## Unidad 6 – Zonas Horarias y Shift/Lag

| Método | Cuándo usarlo |
|--------|--------------|
| `tz_localize('UTC')` | Asignar zona horaria a serie naive |
| `tz_convert('America/Argentina/Buenos_Aires')` | Convertir entre zonas |
| `serie.shift(1)` | Lag: desplaza 1 período hacia adelante |
| `serie.shift(-1)` | Lead: desplaza 1 período hacia atrás |

In [ ]:
# tz_localize → tz_convert
idx_utc = pd.date_range('2023-06-01', periods=6, freq='h', tz='UTC')
serie_utc = pd.Series([100, 110, 105, 120, 115, 130], index=idx_utc, name='temperatura')

serie_arg = serie_utc.tz_convert('America/Argentina/Buenos_Aires')

print('UTC:')
print(serie_utc)
print('\nArgentina (UTC-3):')
print(serie_arg)

In [ ]:
# shift() para features de lag y variación
rng = np.random.default_rng(5)
idx_m = pd.date_range('2022-01', periods=24, freq='ME')
ventas_m = pd.Series(rng.integers(1000, 5000, 24), index=idx_m, name='ventas')

df_lag = pd.DataFrame({'ventas': ventas_m})
df_lag['lag_1']     = df_lag['ventas'].shift(1)
df_lag['lag_2']     = df_lag['ventas'].shift(2)
df_lag['variacion'] = df_lag['ventas'] - df_lag['lag_1']
df_lag['pct_var']   = df_lag['ventas'].pct_change() * 100
print(df_lag.head(8).round(2))

In [ ]:
# Visualizar serie y su lag
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(df_lag.index, df_lag['ventas'], label='Ventas', marker='o', ms=4)
ax.plot(df_lag.index, df_lag['lag_1'],  label='Lag-1',  linestyle='--', alpha=0.7)
ax.set_title('Ventas mensuales y Lag-1')
ax.legend()
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show()

---
## Unidad 7 – Resampling, Rolling, EWMA y Expanding

| Operación | Descripción |
|-----------|-------------|
| `resample('D').sum()` | Agregar a frecuencia diaria |
| `resample('W').mean()` | Media semanal |
| `resample('ME').sum()` | Suma mensual |
| `rolling(7).mean()` | Media móvil de 7 períodos |
| `ewm(span=7).mean()` | Media móvil exponencial |
| `expanding().mean()` | Media acumulada (expanding window) |

In [ ]:
# Serie horaria → resample diario y semanal
rng = np.random.default_rng(6)
idx_h = pd.date_range('2023-01-01', periods=24*90, freq='h')
ventas_h = pd.Series(rng.poisson(50, len(idx_h)), index=idx_h, name='ventas')

ventas_d = ventas_h.resample('D').sum()
ventas_w = ventas_h.resample('W').mean()
ventas_m = ventas_h.resample('ME').sum()

print(f'Horario: {len(ventas_h)} obs | Diario: {len(ventas_d)} | Semanal: {len(ventas_w)} | Mensual: {len(ventas_m)}')
print('\nPrimeras filas diarias:')
print(ventas_d.head())

In [ ]:
# Rolling + EWMA + Expanding
rolling_7  = ventas_d.rolling(window=7).mean()
ewma_7     = ventas_d.ewm(span=7).mean()
expanding_ = ventas_d.expanding().mean()

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(ventas_d.index,  ventas_d,   alpha=0.3, color='gray',   label='Diario')
ax.plot(rolling_7.index, rolling_7,  color='steelblue',          label='Rolling 7d')
ax.plot(ewma_7.index,    ewma_7,     color='orange',             label='EWMA span=7')
ax.plot(expanding_.index,expanding_, color='green',  linestyle=':', label='Expanding')
ax.set_title('Suavizamiento de serie temporal')
ax.legend()
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# Upsample: de mensual a diario con forward fill
ventas_m_up = ventas_m.resample('D').ffill()
print(f'Mensual: {len(ventas_m)} obs → Upsample diario: {len(ventas_m_up)} obs')
print(ventas_m_up.head())

---
## Unidad 8 – GroupBy y Agregaciones

In [ ]:
# Dataset: calidad del aire (sintético inspirado en Pune)
rng = np.random.default_rng(7)
n = 500
df_air = pd.DataFrame({
    'ciudad':    rng.choice(['Buenos Aires','Córdoba','Rosario','Mendoza'], n),
    'mes':       rng.integers(1, 13, n),
    'pm25':      rng.uniform(5, 150, n).round(1),
    'pm10':      rng.uniform(10, 250, n).round(1),
    'no2':       rng.uniform(5, 80, n).round(1),
    'aqi':       rng.integers(20, 300, n),
})
print(df_air.head())

In [ ]:
# GroupBy simple
print('Media AQI por ciudad:')
print(df_air.groupby('ciudad')['aqi'].mean().sort_values(ascending=False).round(2))

In [ ]:
# Múltiples agregaciones
resumen = df_air.groupby('ciudad').agg(
    aqi_media=('aqi', 'mean'),
    aqi_max=('aqi', 'max'),
    pm25_media=('pm25', 'mean'),
    n_registros=('aqi', 'count'),
).round(2)
print(resumen)

In [ ]:
# Visualización GroupBy
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Boxplot AQI por ciudad
df_air.boxplot(column='aqi', by='ciudad', ax=axes[0])
axes[0].set_title('Distribución AQI por ciudad')
axes[0].set_xlabel('')

# Evolución media mensual
pivot = df_air.pivot_table(values='aqi', index='mes', columns='ciudad', aggfunc='mean')
pivot.plot(ax=axes[1], marker='o')
axes[1].set_title('AQI mensual promedio')
axes[1].set_xlabel('Mes')
plt.suptitle('')
plt.tight_layout()
plt.show()

---
## Unidad 9 – Operaciones sobre Strings

pandas expone el accesor `.str` para operaciones vectorizadas sobre columnas de texto.

| Método | Descripción |
|--------|-------------|
| `.str.lower()` / `.str.upper()` | Normalización |
| `.str.strip()` | Eliminar espacios |
| `.str.contains(pat)` | Filtro con regex |
| `.str.replace(pat, repl)` | Reemplazo |
| `.str.split(sep).str[n]` | Dividir y acceder |
| `.str.extract(r'pattern')` | Extraer grupos regex |

In [ ]:
df_str = pd.DataFrame({
    'nombre': ['  Juan Pérez ', 'Ana García', 'CARLOS LOPEZ', 'maria gomez', ' Pedro  '],
    'email':  ['juan@gmail.com','ana@yahoo.com','carlos@empresa.ar','MARIA@GMAIL.COM','pedro@out.com'],
    'codigo': ['PRD-001-A', 'PRD-002-B', 'SRV-003-A', 'SRV-004-C', 'PRD-005-B'],
})

# Normalización
df_str['nombre_clean'] = df_str['nombre'].str.strip().str.title()
df_str['email_clean']  = df_str['email'].str.lower()

# Extraer dominio del email
df_str['dominio'] = df_str['email_clean'].str.split('@').str[1]

# Extraer tipo y número de código
df_str[['tipo','num','cat']] = df_str['codigo'].str.extract(r'([A-Z]+)-(\d+)-([A-Z])')

print(df_str[['nombre_clean','dominio','tipo','num','cat']])

In [ ]:
# Filtros con str.contains
gmail_users = df_str[df_str['email_clean'].str.contains('gmail')]
print('Usuarios con gmail:', gmail_users['nombre_clean'].tolist())

productos = df_str[df_str['codigo'].str.startswith('PRD')]
print('Productos:', productos['codigo'].tolist())

In [ ]:
# Conteo de frecuencia de dominios
print(df_str['dominio'].value_counts())

---
## Unidad 10 – Tipos de Datos, Category y Downcast

| Tipo | Memoria | Cuándo usarlo |
|------|---------|--------------|
| `int64` | 8 bytes | Default entero |
| `int8/int16/int32` | 1-4 bytes | Rango conocido y acotado |
| `float32` | 4 bytes | Precisión no crítica |
| `category` | Variable | Columna con baja cardinalidad |
| `bool` | 1 byte | Flags binarios |

In [ ]:
rng = np.random.default_rng(8)
n = 100_000
df_types = pd.DataFrame({
    'id':       rng.integers(0, 1_000_000, n),
    'edad':     rng.integers(18, 80, n),
    'puntaje':  rng.uniform(0, 100, n),
    'ciudad':   rng.choice(['Buenos Aires','Córdoba','Rosario','Mendoza','Tucumán'], n),
    'activo':   rng.integers(0, 2, n),
})

print('--- Tipos y memoria originales ---')
print(df_types.dtypes)
print(f'Memoria total: {df_types.memory_usage(deep=True).sum() / 1e6:.2f} MB')

In [ ]:
# Optimización de tipos
df_opt = df_types.copy()
df_opt['id']      = pd.to_numeric(df_opt['id'],     downcast='unsigned')
df_opt['edad']    = pd.to_numeric(df_opt['edad'],   downcast='unsigned')
df_opt['puntaje'] = df_opt['puntaje'].astype('float32')
df_opt['ciudad']  = df_opt['ciudad'].astype('category')
df_opt['activo']  = df_opt['activo'].astype('bool')

print('--- Tipos optimizados ---')
print(df_opt.dtypes)
print(f'Memoria optimizada: {df_opt.memory_usage(deep=True).sum() / 1e6:.2f} MB')

In [ ]:
# Conversión a numpy
arr_numpy = df_opt[['id','edad','puntaje']].to_numpy()
print('Tipo numpy:', arr_numpy.dtype)
print('Shape:', arr_numpy.shape)

# category: categorías disponibles
print('\nCategorías ciudad:', df_opt['ciudad'].cat.categories.tolist())
print('Códigos internos (primeros 5):', df_opt['ciudad'].cat.codes.head().tolist())

---
## Unidad 11 – loc, iloc y Máscaras Booleanas

| Selector | Eje | Extremos | Ejemplo |
|----------|-----|----------|---------|
| `loc` | etiquetas | **inclusivo** | `df.loc[0:5, 'col']` |
| `iloc` | posiciones | **exclusivo** final | `df.iloc[0:5, 0]` |
| Máscara | booleana | — | `df[df['col'] > 10]` |

**Operadores booleanos**: `&` (AND), `|` (OR), `~` (NOT). Usar paréntesis.

In [ ]:
rng = np.random.default_rng(9)
df_loc = pd.DataFrame({
    'nombre': [f'Cliente_{i}' for i in range(20)],
    'edad':   rng.integers(18, 70, 20),
    'ventas': rng.uniform(100, 10000, 20).round(2),
    'region': rng.choice(['Norte','Sur','Centro'], 20),
})

# loc – por etiquetas
print('loc filas 3-6, columnas nombre y ventas:')
print(df_loc.loc[3:6, ['nombre','ventas']])

In [ ]:
# iloc – por posición
print('iloc primeras 3 filas, primeras 2 cols:')
print(df_loc.iloc[:3, :2])

In [ ]:
# Máscaras booleanas
# Clientes de Norte con ventas > 5000
mask = (df_loc['region'] == 'Norte') & (df_loc['ventas'] > 5000)
print('Norte + ventas > 5000:')
print(df_loc[mask])

# Clientes jóvenes (<30) O con ventas altas (>8000)
mask2 = (df_loc['edad'] < 30) | (df_loc['ventas'] > 8000)
print(f'\nJóvenes o ventas altas: {mask2.sum()} registros')

# Negación
no_centro = df_loc[~(df_loc['region'] == 'Centro')]
print(f'Fuera de Centro: {len(no_centro)} registros')

In [ ]:
# Asignación condicional con loc
df_loc['segmento'] = 'Regular'
df_loc.loc[df_loc['ventas'] > 7000, 'segmento'] = 'Premium'
df_loc.loc[df_loc['ventas'] < 1000, 'segmento'] = 'Bajo'
print(df_loc['segmento'].value_counts())

---
## Unidad 12 – Transformaciones Vectorizadas

Preferir operaciones vectorizadas sobre `.apply()` por performance.

| Herramienta | Uso |
|-------------|-----|
| `np.where(cond, val_T, val_F)` | Condición binaria |
| `np.select(conds, vals, default)` | Múltiples condiciones |
| `df.assign(col=expr)` | Agregar columna de forma funcional |
| `df.eval('expr')` | Expresiones en strings |

In [ ]:
rng = np.random.default_rng(10)
n = 1000
df_v = pd.DataFrame({
    'quantity': rng.integers(1, 30, n),
    'price':    rng.uniform(10, 500, n).round(2),
})

# np.where – condición binaria
df_v['descuento'] = np.where(df_v['quantity'] > 10, 0.10, 0.0)
df_v['precio_final'] = df_v['price'] * (1 - df_v['descuento'])
print('np.where:')
print(df_v.head())

In [ ]:
# np.select – múltiples condiciones
condiciones = [
    df_v['quantity'] <= 5,
    df_v['quantity'] <= 12,
    df_v['quantity'] <= 20,
]
opciones = ['Unidad', 'Pack', 'Caja']
df_v['tipo_venta'] = np.select(condiciones, opciones, default='Mayorista')
print('\nnp.select:')
print(df_v['tipo_venta'].value_counts())

In [ ]:
# assign – forma funcional (encadenable)
df_v2 = (df_v
    .assign(revenue = lambda d: d['price'] * d['quantity'])
    .assign(revenue_con_dto = lambda d: d['precio_final'] * d['quantity'])
    .assign(ahorro = lambda d: d['revenue'] - d['revenue_con_dto'])
)
print('Totales:')
print(df_v2[['revenue','revenue_con_dto','ahorro']].sum().round(2))

In [ ]:
# eval – expresiones en string (muy eficiente en DataFrames grandes)
df_v['margen'] = df_v.eval('(price - 8) / price * 100')
print('Margen medio:', df_v['margen'].mean().round(2), '%')

---
## Desafío – Análisis de Bitcoin

In [ ]:
# Cargar datos Bitcoin (real o sintético)
try:
    df_btc = pd.read_csv('bitcoin_data.csv', parse_dates=['Date'], index_col='Date')
    print('Dataset Bitcoin cargado.')
except FileNotFoundError:
    rng = np.random.default_rng(11)
    idx_btc = pd.date_range('2020-01-01', periods=365*4, freq='D')
    precio_btc = 10000 * np.cumprod(1 + rng.normal(0.001, 0.04, len(idx_btc)))
    df_btc = pd.DataFrame({'Close': precio_btc, 'Volume': rng.uniform(1e9, 5e10, len(idx_btc))}, index=idx_btc)
    print('Dataset sintético Bitcoin creado.')

print(df_btc.describe().round(2))

In [ ]:
# Rolling 30d + rendimiento diario
df_btc['MA_30']   = df_btc['Close'].rolling(30).mean()
df_btc['retorno'] = df_btc['Close'].pct_change() * 100

fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=True)
axes[0].plot(df_btc.index, df_btc['Close'],  alpha=0.6, label='Precio')
axes[0].plot(df_btc.index, df_btc['MA_30'],  label='MA 30d', linewidth=2)
axes[0].set_title('Bitcoin – Precio y Media Móvil 30 días')
axes[0].legend()

axes[1].bar(df_btc.index, df_btc['retorno'], color=np.where(df_btc['retorno']>=0,'green','red'), alpha=0.5)
axes[1].set_title('Retorno diario (%)')
axes[1].axhline(0, color='black', linewidth=0.5)

plt.tight_layout()
plt.show()

---
## Unidades 13-14 – Matplotlib: Principios y Tipos de Gráficos

### Principios de visualización
- **Claridad**: eliminar chart junk (grillas innecesarias, 3D decorativo).
- **Integridad**: el eje Y debe comenzar en 0 para barras.
- **Contexto**: títulos, etiquetas de ejes, leyendas siempre presentes.
- **Jerarquía visual**: lo más importante tiene más tamaño o contraste.

### Anatomía de una figura
```
Figure → Axes → Artists (Line2D, Rectangle, Text, …)
```

In [ ]:
# 1. Gráfico de líneas – evolución temporal
rng = np.random.default_rng(12)
meses = pd.date_range('2022-01', periods=24, freq='ME')
prod_A = 1000 + np.cumsum(rng.normal(50, 100, 24))
prod_B = 1200 + np.cumsum(rng.normal(30, 80,  24))

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(meses, prod_A, marker='o', ms=5, label='Producto A')
ax.plot(meses, prod_B, marker='s', ms=5, label='Producto B')
ax.set_title('Evolución de ventas mensuales')
ax.set_xlabel('Fecha')
ax.set_ylabel('Unidades')
ax.legend()
ax.tick_params(axis='x', rotation=45)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'{x:,.0f}'))
plt.tight_layout()
plt.show()

In [ ]:
# 2. Scatter con colorbar
n_sc = 200
x_sc = rng.normal(0, 1, n_sc)
y_sc = 2*x_sc + rng.normal(0, 0.5, n_sc)
color_sc = rng.uniform(0, 100, n_sc)

fig, ax = plt.subplots(figsize=(7, 5))
sc = ax.scatter(x_sc, y_sc, c=color_sc, cmap='viridis', alpha=0.7, edgecolors='none')
plt.colorbar(sc, ax=ax, label='Intensidad')
ax.set_title('Scatter con colorbar')
ax.set_xlabel('X')
ax.set_ylabel('Y')
plt.tight_layout()
plt.show()

In [ ]:
# 3. Barras agrupadas
regiones = ['Norte','Sur','Centro','Este']
q1_vals  = rng.integers(400, 900, 4)
q2_vals  = rng.integers(400, 900, 4)
x_pos = np.arange(len(regiones))
w = 0.35

fig, ax = plt.subplots(figsize=(8, 4))
b1 = ax.bar(x_pos - w/2, q1_vals, width=w, label='Q1')
b2 = ax.bar(x_pos + w/2, q2_vals, width=w, label='Q2')
ax.set_xticks(x_pos)
ax.set_xticklabels(regiones)
ax.set_title('Ventas por región y trimestre')
ax.set_ylabel('Ventas')
ax.legend()
ax.bar_label(b1, fmt='%d', padding=2)
ax.bar_label(b2, fmt='%d', padding=2)
plt.tight_layout()
plt.show()

In [ ]:
# 4. Histograma con curva KDE
datos_hist = rng.normal(loc=50, scale=10, size=500)

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(datos_hist, bins=30, density=True, color='steelblue', alpha=0.6, edgecolor='white', label='Histograma')
from scipy.stats import gaussian_kde
kde = gaussian_kde(datos_hist)
x_kde = np.linspace(datos_hist.min(), datos_hist.max(), 200)
ax.plot(x_kde, kde(x_kde), color='red', lw=2, label='KDE')
ax.set_title('Distribución con KDE')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# 5. Boxplot
grupos_box = {
    'Grupo A': rng.normal(60, 10, 100),
    'Grupo B': rng.normal(75, 15, 100),
    'Grupo C': rng.normal(50, 5,  100),
}
fig, ax = plt.subplots(figsize=(7, 4))
ax.boxplot(grupos_box.values(), labels=grupos_box.keys(), patch_artist=True,
           boxprops=dict(facecolor='lightblue'),
           medianprops=dict(color='red', lw=2))
ax.set_title('Comparación de grupos (Boxplot)')
ax.set_ylabel('Valor')
plt.tight_layout()
plt.show()

In [ ]:
# 6. Torta (pie)
labels_pie = ['Electrónica','Ropa','Alimentos','Hogar','Otros']
sizes_pie  = [35, 25, 20, 12, 8]
explode    = [0.05]*5

fig, ax = plt.subplots(figsize=(6, 6))
wedges, texts, autotexts = ax.pie(
    sizes_pie, labels=labels_pie, autopct='%1.1f%%',
    explode=explode, startangle=90,
    colors=sns.color_palette('pastel'),
)
ax.set_title('Participación por categoría')
plt.tight_layout()
plt.show()

In [ ]:
# 7. Múltiples subplots en grid
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

# Subplot 1: línea
axes[0,0].plot(prod_A, color='steelblue')
axes[0,0].set_title('Serie Producto A')

# Subplot 2: barras
axes[0,1].bar(regiones, q1_vals, color='salmon')
axes[0,1].set_title('Ventas Q1 por región')

# Subplot 3: scatter
axes[1,0].scatter(x_sc[:80], y_sc[:80], alpha=0.5, color='green')
axes[1,0].set_title('Scatter simple')

# Subplot 4: hist
axes[1,1].hist(datos_hist, bins=20, color='purple', alpha=0.7)
axes[1,1].set_title('Histograma')

for ax in axes.flat:
    ax.grid(True, alpha=0.3)

plt.suptitle('Dashboard Matplotlib – 4 gráficos', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
## Unidad 15 – Plotly Express: Gráficos Interactivos

Plotly genera gráficos interactivos (hover, zoom, export) en HTML.

| Función | Gráfico |
|---------|---------|
| `px.bar()` | Barras |
| `px.line()` | Líneas |
| `px.scatter()` | Dispersión |
| `px.pie(hole=0.4)` | Donut |
| `px.treemap()` | Treemap |
| `fig.write_html('file.html')` | Exportar |

In [ ]:
if PLOTLY:
    rng = np.random.default_rng(13)
    df_plotly = pd.DataFrame({
        'categoria': ['Electrónica','Ropa','Alimentos','Hogar','Libros',
                      'Deportes','Juguetes','Belleza'],
        'ventas_total': rng.integers(5000, 50000, 8),
        'crecimiento':  rng.uniform(-10, 30, 8).round(1),
    })

    # 1. Bar chart interactivo
    fig_bar = px.bar(
        df_plotly.sort_values('ventas_total', ascending=False),
        x='categoria', y='ventas_total', color='crecimiento',
        color_continuous_scale='RdYlGn',
        title='Ventas por categoría',
        labels={'ventas_total':'Ventas','categoria':'Categoría'},
    )
    fig_bar.show()
else:
    print('Instalar plotly: pip install plotly')

In [ ]:
if PLOTLY:
    # 2. Donut chart
    fig_pie = px.pie(
        df_plotly, names='categoria', values='ventas_total',
        hole=0.4, title='Participación de mercado (Donut)',
        color_discrete_sequence=px.colors.qualitative.Pastel,
    )
    fig_pie.update_traces(textinfo='percent+label')
    fig_pie.show()

In [ ]:
if PLOTLY:
    # 3. Treemap
    df_plotly['sector'] = ['Tech','Moda','Comida','Hogar','Cultura',
                            'Deporte','Juego','Moda']
    fig_tree = px.treemap(
        df_plotly, path=['sector','categoria'],
        values='ventas_total', color='crecimiento',
        color_continuous_scale='Blues',
        title='Treemap – Ventas por sector y categoría',
    )
    fig_tree.show()

In [ ]:
if PLOTLY:
    # 4. Exportar a HTML
    fig_bar.write_html('ventas_plotly.html')
    print('Exportado: ventas_plotly.html')
    print('Abrir con el navegador para interacción completa.')

---
## Unidad 16 – Storytelling con Datos

### Estructura narrativa
1. **Problema / Pregunta de negocio** – ¿Qué queremos entender?
2. **Contexto** – Datos disponibles, período, alcance.
3. **Metodología** – ¿Cómo se analizó?
4. **Hallazgos clave** – 3-5 insights principales.
5. **Recomendaciones** – Acciones concretas derivadas.
6. **Limitaciones** – Sesgos, datos faltantes, supuestos.

### Adaptación al público
| Audiencia | Enfoque |
|-----------|---------|
| Ejecutivo | KPIs, impacto económico, decisiones |
| Técnico | Metodología, métricas, reproducibilidad |
| Mixto | Resumen ejecutivo + apéndice técnico |

In [ ]:
# Reporte ejecutivo simulado – 4 paneles
rng = np.random.default_rng(14)
meses = pd.date_range('2023-01', periods=12, freq='ME')
categorias = ['Premium','Standard','Básico']

ventas_total = np.array([120, 135, 128, 142, 138, 155, 160, 152, 170, 165, 178, 190]) * 1000
churn_rate   = np.array([5.2, 4.8, 5.0, 4.5, 4.3, 4.0, 3.8, 4.1, 3.5, 3.2, 3.0, 2.8])
mix_cat = pd.DataFrame({
    'Mes': meses,
    'Premium':  rng.integers(30, 50, 12),
    'Standard': rng.integers(40, 60, 12),
    'Básico':   rng.integers(20, 40, 12),
}).set_index('Mes')
nps = rng.integers(40, 80, 12)

fig, axes = plt.subplots(2, 2, figsize=(14, 9))
fig.suptitle('Reporte Ejecutivo – 2023\nCrecimiento y Retención de Clientes',
             fontsize=15, fontweight='bold', y=1.01)

# Panel 1: Ventas acumuladas
axes[0,0].fill_between(meses, ventas_total/1e6, alpha=0.3, color='steelblue')
axes[0,0].plot(meses, ventas_total/1e6, 'o-', color='steelblue')
axes[0,0].set_title('Ventas mensuales (M$)')
axes[0,0].set_ylabel('Millones ARS')
axes[0,0].tick_params(axis='x', rotation=45)

# Panel 2: Churn rate
color_churn = ['green' if c < 4.0 else 'red' for c in churn_rate]
axes[0,1].bar(range(12), churn_rate, color=color_churn, alpha=0.7)
axes[0,1].axhline(4.0, color='black', linestyle='--', lw=1, label='Objetivo 4%')
axes[0,1].set_title('Tasa de Churn (%)')
axes[0,1].set_ylabel('%')
axes[0,1].set_xticks(range(12))
axes[0,1].set_xticklabels([m.strftime('%b') for m in meses], rotation=45)
axes[0,1].legend()

# Panel 3: Mix de categorías (stacked bar)
bottom = np.zeros(12)
colors_stack = ['#2196F3','#FF9800','#4CAF50']
for cat, col in zip(categorias, colors_stack):
    axes[1,0].bar(range(12), mix_cat[cat], bottom=bottom, label=cat, color=col, alpha=0.85)
    bottom += mix_cat[cat].values
axes[1,0].set_title('Mix de clientes por categoría')
axes[1,0].set_xticks(range(12))
axes[1,0].set_xticklabels([m.strftime('%b') for m in meses], rotation=45)
axes[1,0].legend(loc='upper left', fontsize=8)

# Panel 4: NPS
axes[1,1].plot(range(12), nps, 'D-', color='purple', ms=7)
axes[1,1].fill_between(range(12), 50, nps,
                        where=(nps >= 50), alpha=0.2, color='green', label='NPS≥50')
axes[1,1].fill_between(range(12), 50, nps,
                        where=(nps < 50), alpha=0.2, color='red', label='NPS<50')
axes[1,1].axhline(50, color='gray', lw=0.8, linestyle='--')
axes[1,1].set_title('Net Promoter Score (NPS)')
axes[1,1].set_xticks(range(12))
axes[1,1].set_xticklabels([m.strftime('%b') for m in meses], rotation=45)
axes[1,1].legend(fontsize=8)

plt.tight_layout()
plt.savefig('reporte_ejecutivo.png', dpi=150, bbox_inches='tight')
plt.show()
print('Reporte guardado como reporte_ejecutivo.png')

### Insights del Reporte

1. **Ventas**: crecimiento del 58% en 2023 (120k → 190k ARS mensuales).
2. **Churn**: cayó del 5.2% en enero al 2.8% en diciembre — la estrategia de retención funciona.
3. **Mix**: clientes Premium aumentaron proporcionalmente → mejora en valor de vida.
4. **NPS**: recuperación en H2 — correlaciona con reducción de churn.

**Recomendación**: mantener la inversión en retención Premium; investigar meses con churn > 4%.

---
## Desafío Final – HR Dataset

In [ ]:
# Cargar HR dataset (real o sintético)
try:
    df_hr = pd.read_csv('HR_dataset.csv')
    print('HR Dataset cargado.')
except FileNotFoundError:
    rng = np.random.default_rng(15)
    n_hr = 800
    df_hr = pd.DataFrame({
        'employee_id': range(1, n_hr+1),
        'departamento': rng.choice(['Ventas','IT','RRHH','Finanzas','Operaciones'], n_hr),
        'salario':      rng.integers(30000, 120000, n_hr),
        'antiguedad':   rng.integers(0, 20, n_hr),
        'satisfaccion': rng.choice([1,2,3,4,5], n_hr),
        'evaluacion':   rng.uniform(0.4, 1.0, n_hr).round(2),
        'promovido':    rng.choice([0, 1], n_hr, p=[0.85, 0.15]),
    })
    print('HR Dataset sintético creado.')

print(df_hr.shape)
print(df_hr.dtypes)
print(df_hr.head())

In [ ]:
# Análisis exploratorio
print('--- Estadísticas descriptivas ---')
print(df_hr.describe().round(2))
print('\n--- Nulos ---')
print(df_hr.isnull().sum())

In [ ]:
# Visualización HR
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Salario por departamento
df_hr.boxplot(column='salario', by='departamento', ax=axes[0,0])
axes[0,0].set_title('Salario por departamento')
axes[0,0].set_xlabel('')

# 2. Satisfacción vs Evaluación
sc = axes[0,1].scatter(
    df_hr['satisfaccion'], df_hr['evaluacion'],
    c=df_hr['promovido'], cmap='coolwarm', alpha=0.5, edgecolors='none')
axes[0,1].set_title('Satisfacción vs Evaluación (color=promovido)')
axes[0,1].set_xlabel('Satisfacción')
axes[0,1].set_ylabel('Evaluación')
plt.colorbar(sc, ax=axes[0,1])

# 3. Distribución de antigüedad
axes[1,0].hist(df_hr['antiguedad'], bins=20, color='teal', edgecolor='white', alpha=0.8)
axes[1,0].set_title('Distribución de antigüedad')
axes[1,0].set_xlabel('Años')

# 4. Tasa de promoción por departamento
promo_rate = df_hr.groupby('departamento')['promovido'].mean() * 100
promo_rate.sort_values(ascending=False).plot(kind='bar', ax=axes[1,1],
    color='steelblue', edgecolor='white')
axes[1,1].set_title('Tasa de promoción por departamento (%)')
axes[1,1].set_ylabel('%')
axes[1,1].tick_params(axis='x', rotation=45)

plt.suptitle('Análisis HR Dataset', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Pipeline de clasificación: predecir promoción
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

features_hr = ['salario','antiguedad','satisfaccion','evaluacion']
X_hr = df_hr[features_hr]
y_hr = df_hr['promovido']

X_tr, X_te, y_tr, y_te = train_test_split(X_hr, y_hr, test_size=0.2, random_state=42)

pipe_hr = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler',  StandardScaler()),
    ('clf',     LogisticRegression(max_iter=1000, class_weight='balanced')),
])

pipe_hr.fit(X_tr, y_tr)
y_pred = pipe_hr.predict(X_te)
print(classification_report(y_te, y_pred, target_names=['No promovido','Promovido']))